# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.
- **`.hrs4`** — Up Condition Pellet stage.
- **`.hrs5`** — Down Condition VNS stage.
- **`.hrs6`** — Up Condition VNS stage.
- **`.hrsft`** — Frequency Test stage.

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # File readers
    read_hrs2, read_hrs3, read_hrs4, read_hrs5, read_hrs6, read_hrs_ft,
    find_hrs_files, detect_app_version,
    # Summary printers
    print_hrs2_summary,
    # HRS2 plots
    plot_amplitude_distribution, plot_background_emg_views, plot_hrs2_analysis,
    plot_actual_trial_timeline, plot_mwave_control_error, plot_frequency_test,
    # SNR analysis
    compute_snr_analysis, compute_mra_snr_analysis,
    plot_hrs2_trials, classify_trials, get_trial_window,
    detect_stim_onset, get_trial_context_window,
    detect_and_correct_failed_trials,
    # Post-hoc global windowing analysis
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
    build_merged_amp_groups,
    make_viewer, compute_h_comparison_data, plot_h_reflex_comparison,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

Helpers loaded.
  App constants: SAMPLE_RATE=5000.0 Hz | BIN=50 ms (250 samples) | TRIAL_RECORD=100 ms
  Stim thresholds: onset >= 4.5 V | end < 1.9 V


# Section 1b: Auto-Detect Recording Files

Set `recording_dir` to the path of your recording folder.  
The `.hrs1` and `.hrs2` files will be found automatically.

In [2]:
# ── Multi-Recording Configuration ─────────────────────────────────────────────
# Each entry: ("Display Label", "relative/path/to/recording_dir", sample_rate_hz)
# sample_rate_hz: explicit Hz (e.g. 10000.0 or 5000.0); None = auto-detect from data.
RECORDING_DIRS = [
    
    # "HRPilot-18_Control/CC1_HRPILOT-18_BOOTH1_500US_6-29-26"
# "HRPilot-17_Control/CC2_HRPILOT-17_BOOTH1_500US_7-2-26"
# "HRPilot-18_Control/CC3_HRPILOT-18_BOOTH1_500US_7-6-26"
# "HRPilot-18_Control/CC3_HRPILOT-18_BOOTH1_500US_7-7-26"
# "HRPilot-18_Control/CC5_HRPILOT-18_BOOTH1_500US_7-8-26"
# "HRPilot-18_Control/CC6_HRPILOT-18_BOOTH1_500US_7-9-26"
    ("HRPILOT-18 CC1",  "HRPilot-18_Control/CC1_HRPILOT-18_BOOTH1_500US_6-29-26",        15000.0),
    ("HRPILOT-18 CC2",  "HRPilot-18_Control/HRPILOT-18_CONTROL1_BOOTH2_250US_10KHZ_7-17-26",         15000.0),
    ("HRPILOT-18 CC3",  "HRPilot-18_Control/HRPILOT-18_CONTROL2_BOOTH1_500US_7-20-26",  15000.0),
    ("HRPILOT-18 CC4",  "HRPilot-18_Control/HRPILOT-18_CONTROL3_BOOTH2_250US_10KHZ_7-21-26",  15000.0),
    ("HRPILOT-18 CC5",  "HRPilot-18_Control/HRPILOT-18_CONTROL4_BOOTH2_250US_10KHZ_7-22-26",  15000.0),
    ("HRPILOT-18 CC6",  "HRPilot-18_Control/HRPILOT-18_CONTROL4_BOOTH2_250US_10KHZ_7-22-26",  15000.0),
        
    
    
    
    ("HRPILOT-18 CTRL1",  "HRPilot-18_Control/HRPILOT-18_M-WAVE_ALGORITHM_TEST_BOOTH1_7-16-26",        15000.0),
    ("HRPILOT-18 CTRL2",  "HRPilot-18_Control/HRPILOT-18_CONTROL1_BOOTH2_250US_10KHZ_7-17-26",         10000.0),
    ("HRPILOT-18 CTRL3",  "HRPilot-18_Control/HRPILOT-18_CONTROL2_BOOTH1_500US_7-20-26",  10000.0),
    ("HRPILOT-18 CTRL4",  "HRPilot-18_Control/HRPILOT-18_CONTROL3_BOOTH2_250US_10KHZ_7-21-26",  10000.0),
    ("HRPILOT-18 CTRL5",  "HRPilot-18_Control/HRPILOT-18_CONTROL4_BOOTH2_250US_10KHZ_7-22-26",  10000.0),
    
    # Add more recordings below — uncomment or append new tuples.
]
# When multiple recordings are loaded a Recording dropdown appears in the selector cell.
print(f"{len(RECORDING_DIRS)} recording(s) configured.")
for _i, (_lbl, _rdir, _rsr) in enumerate(RECORDING_DIRS):
    _sr_str = f"{_rsr} Hz" if _rsr else "auto-detect"
    print(f"  [{_i}] {_lbl!r}  →  {_rdir}  (sample_rate={_sr_str})")

11 recording(s) configured.
  [0] 'HRPILOT-18 CC1'  →  HRPilot-18_Control/CC1_HRPILOT-18_BOOTH1_500US_6-29-26  (sample_rate=15000.0 Hz)
  [1] 'HRPILOT-18 CC2'  →  HRPilot-18_Control/HRPILOT-18_CONTROL1_BOOTH2_250US_10KHZ_7-17-26  (sample_rate=15000.0 Hz)
  [2] 'HRPILOT-18 CC3'  →  HRPilot-18_Control/HRPILOT-18_CONTROL2_BOOTH1_500US_7-20-26  (sample_rate=15000.0 Hz)
  [3] 'HRPILOT-18 CC4'  →  HRPilot-18_Control/HRPILOT-18_CONTROL3_BOOTH2_250US_10KHZ_7-21-26  (sample_rate=15000.0 Hz)
  [4] 'HRPILOT-18 CC5'  →  HRPilot-18_Control/HRPILOT-18_CONTROL4_BOOTH2_250US_10KHZ_7-22-26  (sample_rate=15000.0 Hz)
  [5] 'HRPILOT-18 CC6'  →  HRPilot-18_Control/HRPILOT-18_CONTROL4_BOOTH2_250US_10KHZ_7-22-26  (sample_rate=15000.0 Hz)
  [6] 'HRPILOT-18 CTRL1'  →  HRPilot-18_Control/HRPILOT-18_M-WAVE_ALGORITHM_TEST_BOOTH1_7-16-26  (sample_rate=15000.0 Hz)
  [7] 'HRPILOT-18 CTRL2'  →  HRPilot-18_Control/HRPILOT-18_CONTROL1_BOOTH2_250US_10KHZ_7-17-26  (sample_rate=10000.0 Hz)
  [8] 'HRPILOT-18 CTRL3'  →  HRP

In [3]:
# ── Load all recordings (auto-detects V2 / V3) ─────────────────────────────────
_all_recordings = {}

for (_rlabel, _rdir, _rsr) in RECORDING_DIRS:
    print(f'\n── Loading: {_rlabel!r}  ({_rdir})')
    _rp1, _rp2, _rp3, _rp4, _rp5, _rp6, _rpft = find_hrs_files(_rdir)
    _rav = detect_app_version(_rdir)

    _r_cm_h  = _r_cm_t  = _r_cm_e  = None
    _r_dcp_h = _r_dcp_t = _r_dcp_e = None
    _r_s4_h  = _r_s4_t  = _r_s4_e  = None
    _r_s5_h  = _r_s5_t  = _r_s5_e  = None
    _r_s6_h  = _r_s6_t  = _r_s6_e  = None
    _r_ft_h  = _r_ft_t  = _r_ft_e  = None
    _r_h2h   = _r_h1h   = None
    _r_h2t   = _r_h2e   = []

    # V2/V3: .hrs1 = MH Recruitment Curve, .hrs2 = Control Mode, .hrs3+ = conditioning stages
    if _rp1:
        _r_h2h, _r_h2t, _r_h2e = read_hrs2(_rp1)
        _r_h1h = _r_h2h
        print(f'   .hrs1: {len(_r_h2t)} trials  (MH Recruitment)')
    else:
        print('   .hrs1: not found')
    if _rp2:
        _r_cm_h, _r_cm_t, _r_cm_e = read_hrs2(_rp2)
        print(f'   .hrs2: {len(_r_cm_t)} trials  (Control Mode)')
    else:
        print('   .hrs2: not found')
    if _rp3:
        _r_dcp_h, _r_dcp_t, _r_dcp_e = read_hrs3(_rp3)
        print(f'   .hrs3: {len(_r_dcp_t)} trials  (Down Condition Pellet)')
    if _rav >= 3:
        if _rp4:
            _r_s4_h, _r_s4_t, _r_s4_e = read_hrs4(_rp4)
            print(f'   .hrs4: {len(_r_s4_t)} trials  (Up Condition Pellet)')
        if _rp5:
            _r_s5_h, _r_s5_t, _r_s5_e = read_hrs5(_rp5)
            print(f'   .hrs5: {len(_r_s5_t)} trials  (Down Condition VNS)')
        if _rp6:
            _r_s6_h, _r_s6_t, _r_s6_e = read_hrs6(_rp6)
            print(f'   .hrs6: {len(_r_s6_t)} trials  (Up Condition VNS)')
        if _rpft:
            _r_ft_h, _r_ft_t, _r_ft_e = read_hrs_ft(_rpft)
            print(f'   .hrsft: {len(_r_ft_t)} trials  (Frequency Test)')

    # Control Mode-only: alias as primary analysis when no MH Recruitment stage
    if not _r_h2t and _r_cm_t:
        _r_h2h = _r_cm_h
        _r_h2t = _r_cm_t
        _r_h2e = _r_cm_e
        _r_h1h = _r_h2h
        print('   Note: Control Mode aliased as primary analysis (no MH Recruitment stage).')

    # Resolve sample rate
    _r_detect_sr = _rsr
    if _r_detect_sr is None:
        _r_detect_sr = getattr(_r_h1h, 'sample_rate', None) or 5000.0

    # hrs1_header fallback stub
    if _r_h1h is None:
        _r_sr_val = _r_detect_sr
        class _SampleRateStub:
            sample_rate = _r_sr_val
        _r_h1h = _SampleRateStub()

    # Build stage map for this recording
    _r_sm = {}
    if _r_h2t and (not _r_cm_t or _r_h2t is not _r_cm_t):
        _r_sm['mh_recruitment'] = (_r_h2t, _r_h2h, _r_h2e, 'MH Recruitment Curve (.hrs1)')
    if _r_cm_t:
        _r_sm['control_mode']   = (_r_cm_t,  _r_cm_h,  _r_cm_e,  'Control Mode (.hrs2)')
    if _r_dcp_t:
        _r_sm['dcp']            = (_r_dcp_t, _r_dcp_h, _r_dcp_e, 'Down Condition Pellet (.hrs3)')
    if _r_s4_t:
        _r_sm['up_cond_pellet'] = (_r_s4_t,  _r_s4_h,  _r_s4_e,  'Up Condition Pellet (.hrs4)')
    if _r_s5_t:
        _r_sm['down_cond_vns']  = (_r_s5_t,  _r_s5_h,  _r_s5_e,  'Down Condition VNS (.hrs5)')
    if _r_s6_t:
        _r_sm['up_cond_vns']    = (_r_s6_t,  _r_s6_h,  _r_s6_e,  'Up Condition VNS (.hrs6)')

    _all_recordings[_rlabel] = {
        'stage_map':     _r_sm,
        'sample_rate':   _r_detect_sr,
        'hrs1_header':   _r_h1h,
        'ft_trials':     _r_ft_t,
        'ft_header':     _r_ft_h,
        'app_version':   _rav,
    }
    print(f'   App V{_rav}  |  Stages: {list(_r_sm.keys())}  |  SR: {_r_detect_sr} Hz')

_active_rec_label = next(iter(_all_recordings))
print(f'\n{len(_all_recordings)} recording(s) loaded.  Active: {_active_rec_label!r}')


── Loading: 'HRPILOT-18 CC1'  (HRPilot-18_Control/CC1_HRPILOT-18_BOOTH1_500US_6-29-26)
   .hrs1: 275 trials  (MH Recruitment)
   .hrs2: 201 trials  (Control Mode)
   App V2  |  Stages: ['mh_recruitment', 'control_mode']  |  SR: 15000.0 Hz

── Loading: 'HRPILOT-18 CC2'  (HRPilot-18_Control/HRPILOT-18_CONTROL1_BOOTH2_250US_10KHZ_7-17-26)
   .hrs1: not found
   .hrs2: 1066 trials  (Control Mode)
   Note: Control Mode aliased as primary analysis (no MH Recruitment stage).
   App V2  |  Stages: ['control_mode']  |  SR: 15000.0 Hz

── Loading: 'HRPILOT-18 CC3'  (HRPilot-18_Control/HRPILOT-18_CONTROL2_BOOTH1_500US_7-20-26)
   .hrs1: not found
   .hrs2: 483 trials  (Control Mode)
   Note: Control Mode aliased as primary analysis (no MH Recruitment stage).
   App V3  |  Stages: ['control_mode']  |  SR: 15000.0 Hz

── Loading: 'HRPILOT-18 CC4'  (HRPilot-18_Control/HRPILOT-18_CONTROL3_BOOTH2_250US_10KHZ_7-21-26)
   .hrs1: not found
   .hrs2: 831 trials  (Control Mode)
   Note: Control Mode alias

# Section 3: Peri-Stimulus Trials — MH Recruitment Curve or Control Mode

`hrs2_trials` contains whichever stage was run:
- `.hrs1` present → MH Recruitment Curve trials
- `.hrs1` absent, `.hrs2` present → Control Mode trials (aliased automatically)

In [4]:
# Stage summary for the active recording
print(f'Active recording: {_active_rec_label!r}')
_rec_info = _all_recordings[_active_rec_label]
for _sk, (_st, _sh, _se, _slbl) in _rec_info['stage_map'].items():
    print(f'  {_slbl}: {len(_st)} trials')
print(f'\n(Run the Stage Selector cell below to enable the interactive dropdown.)')

Active recording: 'HRPILOT-18 CC1'
  MH Recruitment Curve (.hrs1): 275 trials
  Control Mode (.hrs2): 201 trials

(Run the Stage Selector cell below to enable the interactive dropdown.)


In [5]:
# ── Recording & Stage Selector ─────────────────────────────────────────────────
# Change the active recording/stage here — all viewer sections update automatically.
from ipywidgets import Dropdown, VBox, Output
from IPython.display import display as _disp

# ── Shared viewer callback registry ────────────────────────────────────────────
if not isinstance(globals().get('_refresh_viewers'), dict):
    _refresh_viewers = {}

def _trigger_refresh():
    for _fn in list(_refresh_viewers.values()):
        try:
            _fn()
        except Exception as _err:
            import traceback
            print(f'[viewer refresh error] {_err}')
            traceback.print_exc()

# ── Active-recording state (module-level vars used by downstream cells) ────────
def _activate_recording(label):
    global _stage_map, recording_sample_rate, hrs1_header, \
           ft_trials, ft_header, ACTIVE_STAGE, \
           _plot_trials, _plot_header, _plot_emg_blocks
    _rec = _all_recordings[label]
    _stage_map            = _rec['stage_map']
    recording_sample_rate = _rec['sample_rate']
    hrs1_header           = _rec['hrs1_header']
    ft_trials             = _rec.get('ft_trials')
    ft_header             = _rec.get('ft_header')
    if globals().get('ACTIVE_STAGE') not in _stage_map:
        ACTIVE_STAGE = next(iter(_stage_map))
    _sel = _stage_map[ACTIVE_STAGE]
    _plot_trials, _plot_header, _plot_emg_blocks = _sel[0], _sel[1], _sel[2]

_activate_recording(_active_rec_label)

# ── Stage dropdown (always shown) ─────────────────────────────────────────────
_sd_opts = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
_stage_drop = Dropdown(options=_sd_opts, value=ACTIVE_STAGE,
                       description='Stage:', layout={'width': '480px'})

def _on_stage_change(change):
    global ACTIVE_STAGE, _plot_trials, _plot_header, _plot_emg_blocks
    ACTIVE_STAGE = _stage_drop.value
    _sel = _stage_map[ACTIVE_STAGE]
    _plot_trials, _plot_header, _plot_emg_blocks = _sel[0], _sel[1], _sel[2]
    _trigger_refresh()

_stage_drop.observe(_on_stage_change, names='value')

# ── Recording dropdown (only if multiple recordings loaded) ────────────────────
if len(_all_recordings) > 1:
    _rec_drop = Dropdown(options=list(_all_recordings.keys()),
                         value=_active_rec_label,
                         description='Recording:', layout={'width': '640px'})

    def _on_rec_change(change):
        global _active_rec_label
        _active_rec_label = _rec_drop.value
        _activate_recording(_active_rec_label)
        _stage_drop.unobserve(_on_stage_change, names='value')
        _stage_drop.options = [(lbl, sk) for sk, (_t, _h, _e, lbl) in _stage_map.items() if _t]
        _stage_drop.value   = ACTIVE_STAGE
        _stage_drop.observe(_on_stage_change, names='value')
        _trigger_refresh()

    _rec_drop.observe(_on_rec_change, names='value')
    _disp(VBox([_rec_drop, _stage_drop]))
else:
    _disp(_stage_drop)

# ── Status ─────────────────────────────────────────────────────────────────────
print(f'Active recording : {_active_rec_label!r}  (App V{_all_recordings[_active_rec_label]["app_version"]})')
print(f'Available stages ({len(_stage_map)}):')
for _k, (_t, _h, _e, _lbl) in _stage_map.items():
    _mark = '  ◄ active' if _k == ACTIVE_STAGE else ''
    print(f'  {_k!r:<22} → {_lbl}  ({len(_t)} trials){_mark}')
print(f'Sample rate : {recording_sample_rate} Hz')

# ── Stimulation Intensity Histogram (viewer section) ──────────────────────────
_hist_out = Output()

def _hist_refresh():
    _st, _sh, _se, _slbl = _stage_map[ACTIVE_STAGE]
    with _hist_out:
        _hist_out.clear_output(wait=True)
        print(f'\nHistogram: {_slbl}  ({len(_st)} trials)  [{_active_rec_label}]')
        plot_amplitude_distribution(_st, _sh)

_refresh_viewers['histogram'] = _hist_refresh
_disp(_hist_out)
_hist_refresh()

Active recording : 'HRPILOT-18 CC1'  (App V2)
Available stages (2):
  'mh_recruitment'       → MH Recruitment Curve (.hrs1)  (275 trials)  ◄ active
  'control_mode'         → Control Mode (.hrs2)  (201 trials)
Sample rate : 15000.0 Hz


Output()

In [6]:
# ── Trial Timeline ─────────────────────────────────────────────────────────────
from ipywidgets import Output
from IPython.display import display as _disp

_tl_out = Output()

def _tl_refresh():
    _st, _sh, _se, _slbl = _stage_map[ACTIVE_STAGE]
    with _tl_out:
        _tl_out.clear_output(wait=True)
        print(f'\n── Trial Timeline: {_slbl}  ({len(_st)} trials)  [{_active_rec_label}]')
        plot_actual_trial_timeline(_st, header=_sh)

_refresh_viewers['trial_timeline'] = _tl_refresh
_disp(_tl_out)
_tl_refresh()

Output()

# Section 3a: Initiation Thresholds Summary

Displays the EMG amplitude thresholds and timing windows used to initiate trials.

**From the recording header:** `trial_initiation_uv_min` and `trial_initiation_uv_max` set the band the background EMG grand mean must fall within to trigger a trial.

**Per-trial:** The active stage's per-trial thresholds (stored with each trial) are shown below if available.

In [7]:
# ── Initiation Thresholds Summary ─────────────────────────────────────────────
from ipywidgets import Output
from IPython.display import display as _disp

_thr_out = Output()

def _thr_refresh():
    _st, _sh, _se, _slbl = _stage_map[ACTIVE_STAGE]
    with _thr_out:
        _thr_out.clear_output(wait=True)
        print("=" * 62)
        print(f"  INITIATION THRESHOLDS — {_active_rec_label}")
        print("=" * 62)

        print("\nRecording Header Thresholds")
        if hasattr(hrs1_header, 'trial_initiation_uv_min'):
            print(f"  Lower bound : {hrs1_header.trial_initiation_uv_min:.2f} µV")
            print(f"  Upper bound : {hrs1_header.trial_initiation_uv_max:.2f} µV")
            if hasattr(hrs1_header, 'trial_initiation_phase_min_ms'):
                print(f"  Monitoring  : {hrs1_header.trial_initiation_phase_min_ms}–"
                      f"{hrs1_header.trial_initiation_phase_max_ms} ms  |  Bin: {hrs1_header.bin_duration_ms} ms")
        else:
            print("  (No header thresholds available — .hrs1 not found)")

        print(f"\nActive Stage: {_slbl}")
        if _sh is None or not _st:
            print("  No trials loaded.")
        elif not hasattr(_st[0], 'min_initiation_threshold'):
            print("  (Initiation thresholds not stored per-trial for this stage type.)")
        else:
            s2_mins = [t.min_initiation_threshold for t in _st]
            s2_maxs = [t.max_initiation_threshold for t in _st]
            all_same = (len(set(round(v, 4) for v in s2_mins)) == 1 and
                        len(set(round(v, 4) for v in s2_maxs)) == 1)
            if all_same:
                print(f"  Lower bound : {s2_mins[0]:.4f} µV  (constant across all {len(_st)} trials)")
                print(f"  Upper bound : {s2_maxs[0]:.4f} µV  (constant across all {len(_st)} trials)")
            else:
                print(f"  Thresholds varied across {len(_st)} trials:")
                print(f"  {'Trial':>6}  {'Lower (µV)':>12}  {'Upper (µV)':>12}")
                for i, (mn, mx) in enumerate(zip(s2_mins, s2_maxs)):
                    print(f"  {i+1:>6}  {mn:>12.4f}  {mx:>12.4f}")
        print("\n" + "=" * 62)

_refresh_viewers['initiation_thresholds'] = _thr_refresh
_disp(_thr_out)
_thr_refresh()

Output()

In [8]:
shared_ylim = (-1500, 1500)  # Set a common y-axis limit for all trial plots

In [9]:
#  Configuration 
PRE_PLOT_MS  = 2   # ms before stim onset to display
POST_PLOT_MS = 15  # ms after  stim onset to display
N_PER_PAGE   = 6   # trials shown per page (2 rows x 3 cols)

# M/H wave window constants (ms relative to stim onset)
M_WAVE_START_MS = 1.8 
M_WAVE_END_MS   = 4.5
H_WAVE_START_MS = 6.5
H_WAVE_END_MS   = 9

PRE_AVG_MS  = 2   # ms before stim onset
POST_AVG_MS = 15  # ms after  stim onset
N_PER_PAGE  = 6   # amplitude groups per page (2 rows x 3 cols)


In [10]:
'''# ---- Failed Trial Detector & Corrected Trial Windowing ----
# Classifies all trials for ADC-sync failures, realigns each failed trial to the
# true stim onset found via the first ADC pulse in the continuous context window,
# and plots original (gray) vs corrected (black) waveforms.
# Returns trial_report, failed, passed, realigned for use in later cells.
trial_report, failed, passed, realigned = detect_and_correct_failed_trials(
    _plot_trials, _plot_header, _plot_emg_blocks,
    pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)'''

'# ---- Failed Trial Detector & Corrected Trial Windowing ----\n# Classifies all trials for ADC-sync failures, realigns each failed trial to the\n# true stim onset found via the first ADC pulse in the continuous context window,\n# and plots original (gray) vs corrected (black) waveforms.\n# Returns trial_report, failed, passed, realigned for use in later cells.\ntrial_report, failed, passed, realigned = detect_and_correct_failed_trials(\n    _plot_trials, _plot_header, _plot_emg_blocks,\n    pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,\n    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,\n    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,\n    sample_rate=recording_sample_rate or hrs1_header.sample_rate,\n)'

In [11]:
# ── Optional: Merged Amplitude Group Analysis ─────────────────────────────────
# MERGE_ALL = True  →  collapse every amplitude into a single group.
#
# MERGED_GROUPS accepts two styles (mix freely):
#   Explicit list : [0.12, 0.13, 0.15]   — merge exactly those amplitudes
#   Range tuple   : (0.10, 0.50)         — merge all amplitudes where low <= amp <= high
#
# Examples:
#   MERGED_GROUPS = [[0.12, 0.13], [0.15, 0.16, 0.17]]   ← two explicit groups
#   MERGED_GROUPS = [(0.10, 0.20), (0.25, 0.40)]          ← two range groups
#   MERGED_GROUPS = [(0.10, 0.20), [0.50, 0.55]]          ← range + explicit, mixed
#
# Leave both flags at their defaults to use the standard (unmerged) grouping.
MERGE_ALL     = True   # True → collapse every amplitude into one group
MERGED_GROUPS = []

def _resolve_groups(trials, groups):
    _all_amps = sorted({t.stimulation_amplitude_ma for t in trials})
    resolved = []
    for g in groups:
        if isinstance(g, tuple) and len(g) == 2:
            lo, hi = float(g[0]), float(g[1])
            matched = [a for a in _all_amps if lo <= a <= hi]
            if matched:
                resolved.append(matched)
            else:
                print(f'Warning: range ({lo}, {hi}) matched no amplitudes — skipped.')
        else:
            resolved.append(list(g))
    return resolved

def _apply_merge(trials):
    if MERGE_ALL:
        _amps = sorted({t.stimulation_amplitude_ma for t in trials})
        return build_merged_amp_groups(trials, [_amps])
    if MERGED_GROUPS:
        return build_merged_amp_groups(trials, _resolve_groups(trials, MERGED_GROUPS))
    return trials

print("Merge config: MERGE_ALL =", MERGE_ALL, " | MERGED_GROUPS =", MERGED_GROUPS or "(none)")
print("Re-run viewer cells or change Recording/Stage to apply new merge settings.")

Merge config: MERGE_ALL = True  | MERGED_GROUPS = (none)
Re-run viewer cells or change Recording/Stage to apply new merge settings.


In [22]:
# ── Pre-compute H-Reflex Comparison Data ──────────────────────────────────────────────
# Computed once here for instant rendering in the comparison plot below.
# Re-run this cell if you change PRE_AVG_MS, POST_AVG_MS, or H-wave window constants.
_xr_cache = compute_h_comparison_data(
    _all_recordings, PRE_AVG_MS, POST_AVG_MS, H_WAVE_START_MS, H_WAVE_END_MS
)
_xr_stages = {}
for _rd in _all_recordings.values():
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        if _st and _sk not in _xr_stages:
            _xr_stages[_sk] = _slbl
print(f'H-reflex comparison data pre-computed for {len(_xr_cache)} recording(s), '
      f'{len(_xr_stages)} stage(s): {list(_xr_stages.keys())}')

H-reflex comparison data pre-computed for 11 recording(s), 2 stage(s): ['mh_recruitment', 'control_mode']


In [21]:
# ── HRS2 Analysis: Interactive Averaged Waveforms + Recruitment Curve ─────────
from ipywidgets import Output
from IPython.display import display as _disp

_ana_out = Output()

def _ana_refresh():
    _st, _sh, _se, _slbl = _stage_map[ACTIVE_STAGE]
    _tp = _apply_merge(_st)
    with _ana_out:
        _ana_out.clear_output(wait=True)
        print(f'\n── Analysis: {_slbl}  ({len(_st)} trials)  [{_active_rec_label}]')
        plot_hrs2_analysis(
            _tp, _sh,
            pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
            n_per_page=N_PER_PAGE,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            sample_rate=recording_sample_rate or hrs1_header.sample_rate,
            emg_blocks=_se,
        )

_refresh_viewers['hrs2_analysis'] = _ana_refresh
_disp(_ana_out)
_ana_refresh()

Output()

Merged group ['0.159', '0.160', '0.161', '0.162', '0.163', '0.164', '0.165', '0.166', '0.167', '0.168', '0.169', '0.170', '0.171', '0.172', '0.173', '0.174', '0.175', '0.176', '0.177', '0.178', '0.179', '0.180', '0.181', '0.182', '0.183', '0.184', '0.185', '0.186', '0.187', '0.188', '0.189', '0.190', '0.191', '0.192', '0.193', '0.194', '0.195', '0.196', '0.197', '0.198', '0.199', '0.200', '0.201', '0.202', '0.203', '0.204', '0.205', '0.206', '0.207', '0.208', '0.209', '0.210', '0.211', '0.212', '0.213', '0.214', '0.215', '0.216', '0.217', '0.218', '0.219', '0.220', '0.221', '0.222', '0.223', '0.224', '0.225', '0.226', '0.227', '0.228', '0.229', '0.230'] mA -> avg 0.194 mA, 622 trials


# Section 6b: H-Reflex Size Across Recordings

**H-reflex size per amplitude group** = MRA of the averaged bipolar waveform in the H-wave window, minus the pre-stimulus background MRA:

> **size (µV) = mean|avg_bip(t ∈ [H_START, H_END])| − mean|avg_bip(t < 0)|**

One value per amplitude group; the box shows spread across amplitude groups.

In [20]:
# ── H-Reflex Size Across Recordings: Comparison Plot ─────────────────────────────
from ipywidgets import Output
from IPython.display import display as _disp

_xr_out = Output()

def _xr_refresh():
    with _xr_out:
        _xr_out.clear_output(wait=True)
        plot_h_reflex_comparison(_xr_cache, RECORDING_DIRS, ACTIVE_STAGE, _xr_stages)

_refresh_viewers['h_comparison'] = _xr_refresh
_disp(_xr_out)
_xr_refresh()

Output()

In [15]:
# ── HRS2 Trial Viewer: Interactive Per-Trial Grid + Zoom ──────────────────────
from ipywidgets import Output
from IPython.display import display as _disp

_trv_out = Output()

def _trv_refresh():
    _st, _sh, _se, _slbl = _stage_map[ACTIVE_STAGE]
    _tp = _apply_merge(_st)
    with _trv_out:
        _trv_out.clear_output(wait=True)
        print(f'\n── Trial Viewer: {_slbl}  ({len(_st)} trials)  [{_active_rec_label}]')
        plot_hrs2_trials(
            _tp, _sh,
            pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
            n_per_page=N_PER_PAGE,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            sample_rate=recording_sample_rate or hrs1_header.sample_rate,
            emg_blocks=_se,
        )

_refresh_viewers['trial_viewer'] = _trv_refresh
_disp(_trv_out)
_trv_refresh()

Output()

Merged group ['0.100', '0.105', '0.110', '0.115', '0.120', '0.125', '0.130', '0.135', '0.140', '0.145', '0.150', '0.155', '0.160', '0.165', '0.170', '0.175', '0.180', '0.185', '0.190', '0.195', '0.200', '0.205', '0.210', '0.215', '0.220'] mA -> avg 0.160 mA, 275 trials


# Section 3c: H:M Ratio Summary

Box plot and histogram of H:M ratio for each stimulation polarity group.
If both normal and reversed polarities were used in this session, each group is analysed separately.

In [16]:
# ── Stim Polarity Analysis ────────────────────────────────────────────────────
from ipywidgets import Output
from IPython.display import display as _disp

_pol_out = Output()

def _pol_refresh():
    _st, _sh, _se, _slbl = _stage_map[ACTIVE_STAGE]
    with _pol_out:
        _pol_out.clear_output(wait=True)
        print(f'\n── Polarity / H:M Ratio: {_slbl}  ({len(_st)} trials)  [{_active_rec_label}]')
        trials_by_polarity = split_trials_by_polarity(_st)
        plot_hm_ratio_summary(
            trials_by_polarity, _sh,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            sample_rate=recording_sample_rate or hrs1_header.sample_rate,
            pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
        )
        plot_hwave_regression(
            _st, _se,
            m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            sample_rate=recording_sample_rate or hrs1_header.sample_rate,
            pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
        )

_refresh_viewers['stim_polarity'] = _pol_refresh
_disp(_pol_out)
_pol_refresh()

Output()

# Section 3d: M-Wave Stabilization Control Error (V3)

Trial-by-trial plot of the M-wave stabilization controller output.
- **Left axis** — `m_wave_error` (µV); falls back to `m_wave_window_median` when controller inactive.
- **Right axis** — `stimulation_amplitude_ma` (mA, orange).

Requires V3 recordings (S2 file_version ≥ 9; S4/S5/S6 file_version ≥ 2).

In [17]:
# ── M-Wave Control Error (V3 stages with closed-loop M-wave stabilization) ────
from ipywidgets import Output
from IPython.display import display as _disp

_mw_out = Output()

def _mw_refresh():
    _mw_stages = {k: v for k, v in _stage_map.items()
                  if v[0] and any(
                      not (getattr(t, 'm_wave_error', float('nan')) != getattr(t, 'm_wave_error', float('nan')))
                      or not (getattr(t, 'm_wave_window_median', float('nan')) != getattr(t, 'm_wave_window_median', float('nan')))
                      for t in v[0])}
    with _mw_out:
        _mw_out.clear_output(wait=True)
        if not _mw_stages:
            print("No M-wave stabilization data available in any loaded stage "
                  "(requires V3 S2 file_version ≥ 9, or S4/S5/S6 file_version ≥ 2).")
            return
        _key = ACTIVE_STAGE if ACTIVE_STAGE in _mw_stages else next(iter(_mw_stages))
        _st, _sh, _se, _slbl = _mw_stages[_key]
        print(f'\n── M-Wave Control Error: {_slbl}  ({len(_st)} trials)  [{_active_rec_label}]')
        plot_mwave_control_error(_st, _sh)

_refresh_viewers['mwave_control_error'] = _mw_refresh
_disp(_mw_out)
_mw_refresh()

Output()

# Section 3e: Frequency Test Analysis (V3 FT)

Per-pulse H-wave and M-wave MRA (mean ± std across trials), showing homosynaptic depression across a stimulus train.

In [18]:
# ── Frequency Test Analysis (V3 FT stage) ─────────────────────────────────────
from ipywidgets import Output
from IPython.display import display as _disp

_ft_out = Output()

def _ft_refresh():
    with _ft_out:
        _ft_out.clear_output(wait=True)
        if ft_trials:
            print(f"Frequency Test: {len(ft_trials)} trials  |  "
                  f"{getattr(ft_header, 'n_pulses_per_train', '?')} pulses/train  |  "
                  f"{round(1e6 / ft_header.event_period_us, 1) if getattr(ft_header, 'event_period_us', 0) else '?'} Hz  "
                  f"[{_active_rec_label}]")
            plot_frequency_test(ft_trials, ft_header)
        else:
            print("No Frequency Test data loaded (.hrsft file not found in recording directory).")

_refresh_viewers['frequency_test'] = _ft_refresh
_disp(_ft_out)
_ft_refresh()

Output()

In [19]:
# ── Averaged Waveforms Grouped by Stimulation Amplitude ──────────────────────
from collections import defaultdict
from ipywidgets import Output
from IPython.display import display as _disp

PRE_AVG_MS  = 2   # ms before stim onset
POST_AVG_MS = 15  # ms after  stim onset
M_WAVE_START_MS_AVG = 2.5
M_WAVE_END_MS_AVG   = 4.0
H_WAVE_START_MS_AVG = 6.0
H_WAVE_END_MS_AVG   = 10.0

def _plot_avg_waveforms(_st, _se, _slbl):
    print(f'\n── Averaged Waveforms: {_slbl}  ({len(_st)} trials)  [{_active_rec_label}]')
    groups = defaultdict(list)
    for trial in _st:
        key = round(trial.stimulation_amplitude_ma, 2)
        t_ms_win, emg_win, _, stim_end, _ = get_trial_window(
            trial, PRE_AVG_MS, POST_AVG_MS,
            ms_per_sample=1000.0 / recording_sample_rate)
        groups[key].append((t_ms_win, emg_win, stim_end))
    if not groups:
        print('No trials to plot.')
        return
    _all_emg = np.concatenate([emg for windows in groups.values() for _, emg, _ in windows])
    _lo, _hi = float(np.nanmin(_all_emg)), float(np.nanmax(_all_emg))
    _pad = max(0.08 * (_hi - _lo), 1.0)
    shared_ylim = (_lo - _pad, _hi + _pad)
    _text_offset = 0.05 * (shared_ylim[1] - shared_ylim[0])
    for amp in sorted(groups.keys()):
        windows = groups[amp]
        t_ref = windows[0][0]
        n_pts = len(t_ref)
        padded = np.full((len(windows), n_pts), np.nan)
        for k, (_, emg, _se2) in enumerate(windows):
            n = min(len(emg), n_pts)
            padded[k, :n] = emg[:n]
        avg = np.nanmean(padded, axis=0)
        stim_ends = [se for _, _, se in windows if se is not None]
        mean_stim_end = float(np.mean(stim_ends)) if stim_ends else 0.5
        plt.figure(figsize=(10, 5))
        for row in padded:
            plt.plot(t_ref, row, color='red', alpha=0.6)
        plt.plot(t_ref, avg, color='black', linewidth=2, label='Average EMG')
        plt.axvspan(0, mean_stim_end, color='red', alpha=0.20,
                    label=f'Stim period (0–{mean_stim_end:.1f} ms)')
        plt.axvline(x=0, color='red', linestyle='--', label='Stim onset')
        plt.axvline(x=mean_stim_end, color='red', linestyle='--',
                    label=f'Stim end (~{mean_stim_end:.1f} ms)')
        plt.axvspan(M_WAVE_START_MS_AVG, M_WAVE_END_MS_AVG, color='blue',  alpha=0.3)
        plt.axvspan(H_WAVE_START_MS_AVG, H_WAVE_END_MS_AVG, color='green', alpha=0.3)
        m_mask = (t_ref >= M_WAVE_START_MS_AVG) & (t_ref <= M_WAVE_END_MS_AVG)
        m_t, m_emg = t_ref[m_mask], avg[m_mask]
        h_mask = (t_ref >= H_WAVE_START_MS_AVG) & (t_ref <= H_WAVE_END_MS_AVG)
        h_t, h_emg = t_ref[h_mask], avg[h_mask]
        if len(m_t) and len(h_t):
            m_pi = np.argmax(m_emg)
            h_pi = np.argmax(h_emg)
            plt.axvline(x=m_t[m_pi], color='blue', linestyle=':', linewidth=2,
                        label=f'M-Wave Peak: {m_emg[m_pi]:.1f} µV at {m_t[m_pi]:.2f} ms')
            plt.axvline(x=h_t[h_pi], color='green', linestyle=':', linewidth=2,
                        label=f'H-Wave Peak: {h_emg[h_pi]:.1f} µV at {h_t[h_pi]:.2f} ms')
            plt.text(m_t[m_pi], m_emg[m_pi] + _text_offset, f'{m_emg[m_pi]:.1f} µV',
                     color='blue', fontsize=9, ha='center')
            plt.text(h_t[h_pi], h_emg[h_pi] + _text_offset, f'{h_emg[h_pi]:.1f} µV',
                     color='green', fontsize=9, ha='center')
        plt.title(f'Averaged Waveforms [{_slbl}] (n={len(windows)}) | Stim Amp: {amp:.2f} mA', fontsize=20)
        plt.xlabel('Time (ms)')
        plt.ylabel('Amplitude (µV)')
        plt.grid(True)
        plt.legend(fontsize=7, loc='upper left', frameon=True, framealpha=0.8)
        plt.ylim(-1000, 1500)
        plt.xticks(np.arange(int(np.floor(t_ref[0])), int(np.ceil(t_ref[-1])) + 1, 1))
        plt.tight_layout()
        plt.show()

_avw_out = Output()

def _avw_refresh():
    _st, _sh, _se, _slbl = _stage_map[ACTIVE_STAGE]
    with _avw_out:
        _avw_out.clear_output(wait=True)
        _plot_avg_waveforms(_st, _se, _slbl)

_refresh_viewers['avg_waveforms'] = _avw_refresh
_disp(_avw_out)
_avw_refresh()

Output()